# Mejoras YOLOv10-S - dataset pills-sxdht

Partimos del baseline (`baseline_yolov10.ipynb`): en test mAP50 = 0.985 y mAP50-95 = 0.902.

El analisis TIDE del baseline dice que casi todo lo que falta del mAP50-95 es **localizacion** (+0.08 si se arreglara): el modelo encuentra las pastillas y acierta la clase, pero las cajas no estan del todo ajustadas con IoU de 0.8 a 0.95. Las pastillas pequeñas son las peor ajustadas, y el error de las cajas es aleatorio (no estan desplazadas siempre hacia el mismo lado).

Por eso todos los experimentos atacan la localizacion:

| exp | cambio | por que |
|---|---|---|
| E1 | `epochs=200` | en val el mAP50-95 seguia subiendo en la epoca 100 |
| E2 | `box=10, dfl=2` (por defecto 7.5 y 1.5) | mas peso a la parte de la loss que ajusta la caja |
| E3 | `imgsz=960` (por defecto 640) | las pastillas pequeñas son las peor ajustadas, a mas resolucion mas pixeles por pastilla |
| E4 | TTA: imagen normal + volteada, fusionadas con WBF | como el error de las cajas es aleatorio, hacer la media de dos predicciones deberia reducirlo. No hace falta entrenar |

Cada experimento cambia **una sola cosa** respecto al baseline, para saber que funciona. Elegimos mirando val y al final entrenamos un modelo que junta lo que mejoro. El test solo lo usamos para el baseline y el modelo final.

En Colab: GPU T4. Hay que subir `mapviz.py` y el best.pt del baseline renombrado a `baseline.pt`.

In [ ]:
!pip install -q ultralytics datasets ensemble_boxes

In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from IPython.display import Image, display
import mapviz as mv

SEED = 0
DATA_DIR = Path("data/pills_yolo")

## Dataset

Igual que en el baseline: las mismas 7 clases y las mismas imagenes.

In [ ]:
ds = load_dataset("Francesco/pills-sxdht")

# nombres de las clases (cambia segun la version de datasets)
objs = ds["train"].features["objects"]
if hasattr(objs, "feature"):
    clases = objs.feature["category"].names  # datasets 3.x
else:
    clases = objs["category"].feature.names  # datasets 4.x (colab)

print(ds)
print(clases)

In [ ]:
import shutil

quitar = ["pills", "Ibuphil 600 mg"]
clases_orig = clases
clases = [n for n in clases_orig if n not in quitar]
nuevo_id = {clases_orig.index(n): i for i, n in enumerate(clases)}  # id original -> id nuevo
print(clases)
print(nuevo_id)

shutil.rmtree(DATA_DIR, ignore_errors=True)  # por si habia una conversion anterior

splits = {"train": "train", "validation": "val", "test": "test"}

for split_hf, split_yolo in splits.items():
    (DATA_DIR / "images" / split_yolo).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "labels" / split_yolo).mkdir(parents=True, exist_ok=True)

    n = 0
    for ej in ds[split_hf]:
        cats = ej["objects"]["category"]
        if any(clases_orig[c] in quitar for c in cats):
            print("  quitamos image_id", ej["image_id"], "de", split_hf)
            continue

        nombre = str(ej["image_id"])
        ej["image"].convert("RGB").save(DATA_DIR / "images" / split_yolo / (nombre + ".jpg"))

        W, H = ej["width"], ej["height"]
        lineas = []
        for (x, y, w, h), c in zip(ej["objects"]["bbox"], cats):
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            lineas.append(f"{nuevo_id[c]} {cx} {cy} {w / W} {h / H}")
        with open(DATA_DIR / "labels" / split_yolo / (nombre + ".txt"), "w") as f:
            f.write("\n".join(lineas))
        n += 1

    print(split_hf, n, "imagenes")

# yaml del dataset
with open(DATA_DIR / "data.yaml", "w") as f:
    f.write(f"path: {DATA_DIR.resolve().as_posix()}\n")
    f.write("train: images/train\nval: images/val\ntest: images/test\n")
    f.write("names:\n")
    for i, n in enumerate(clases):
        f.write(f"  {i}: '{n}'\n")

print(open(DATA_DIR / "data.yaml").read())

## Baseline de referencia

No lo reentrenamos, cargamos su best.pt. Lo evaluamos en val en la misma sesion que los experimentos para que los tiempos se puedan comparar.

In [ ]:
resultados = {}

def evaluar(nombre, modelo):
    # el modelo recuerda el imgsz con el que se entreno, no hace falta pasarlo
    m = modelo.val(data=str(DATA_DIR / "data.yaml"), split="val", name=nombre + "_val")
    resultados[nombre] = [m.box.map50, m.box.map, m.speed["inference"]]
    print(nombre, "mAP50:", round(m.box.map50, 3), "mAP50-95:", round(m.box.map, 3))

baseline = YOLO("baseline.pt")
evaluar("baseline", baseline)

## E1: 200 epocas

In [ ]:
e1 = YOLO("yolov10s.pt")
e1.train(data=str(DATA_DIR / "data.yaml"), seed=SEED, name="e1_epochs200", epochs=200)
carpeta_e1 = Path(e1.trainer.save_dir)

e1 = YOLO(carpeta_e1 / "weights" / "best.pt")
evaluar("E1 epochs=200", e1)

# curvas: ver si con 200 epocas ya se estabiliza
display(Image(filename=carpeta_e1 / "results.png", width=900))

## E2: mas peso a la loss de la caja

In [ ]:
e2 = YOLO("yolov10s.pt")
e2.train(data=str(DATA_DIR / "data.yaml"), seed=SEED, name="e2_box10_dfl2", box=10, dfl=2)

e2 = YOLO(Path(e2.trainer.save_dir) / "weights" / "best.pt")
evaluar("E2 box=10 dfl=2", e2)

## E3: mas resolucion

In [ ]:
e3 = YOLO("yolov10s.pt")
e3.train(data=str(DATA_DIR / "data.yaml"), seed=SEED, name="e3_imgsz960", imgsz=960)

e3 = YOLO(Path(e3.trainer.save_dir) / "weights" / "best.pt")
evaluar("E3 imgsz=960", e3)

## Comparacion en val

Ojo: val solo tiene 45 imagenes. En el baseline el mAP50-95 de val sube y baja unas 0.01 entre epocas sin cambiar nada, asi que diferencias mas pequeñas que eso pueden ser casualidad.

In [ ]:
tabla = pd.DataFrame(resultados, index=["mAP50", "mAP50-95", "ms/img"]).T
tabla["dif mAP50-95"] = tabla["mAP50-95"] - tabla.loc["baseline", "mAP50-95"]
print(tabla.round(3))

dif = tabla["dif mAP50-95"].drop("baseline")
plt.barh(dif.index, dif.values)
plt.axvline(0, color="black", lw=1)
plt.xlabel("mAP50-95 en val respecto al baseline")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## E4: TTA con WBF

YOLOv10 en ultralytics no permite `augment=True` (dice que no lo soporta), asi que lo hacemos a mano:
1. Predecimos la imagen normal y la imagen volteada en horizontal.
2. Las cajas de la volteada las volvemos a dar la vuelta para que esten en su sitio.
3. Las juntamos con WBF (*weighted boxes fusion*): las cajas de las dos predicciones que caen sobre la misma pastilla se fusionan en una haciendo la media de sus coordenadas (pesada por el score).

Como las cajas son nuestras, no podemos evaluar con ultralytics. Usamos `mapviz` (el del analisis TIDE) para las dos versiones, sin TTA y con TTA, asi la comparacion es justa.

Ojo: `mapviz` da un poco menos que ultralytics (en val, 0.909 frente a 0.935 de mAP50-95). Es porque `predict` se queda con una sola clase por caja y `val` de ultralytics guarda varias clases para la misma caja, lo que le da mas oportunidades de acertar (p. ej. en la imagen 1 de val dice `white` con 0.84, pero tambien `Xyzall 5mg` con 0.03, que es la buena). Para comparar sin TTA y con TTA da igual, porque las dos se miden igual.

In [ ]:
def predecir(modelo, split, tta=False):
    # devuelve las cajas reales y las predichas en el formato de mapviz
    gts, preds = [], []
    for f in sorted((DATA_DIR / "images" / split).glob("*.jpg")):
        img = cv2.imread(str(f))
        H, W = img.shape[:2]

        for linea in open(DATA_DIR / "labels" / split / (f.stem + ".txt")).read().splitlines():
            c, cx, cy, w, h = linea.split()
            cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
            gts.append(dict(img=f.stem, cls=clases[int(c)], box=(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)))

        r = modelo.predict(img, conf=0.001, verbose=False)[0]
        cajas = r.boxes.xyxyn.cpu().numpy()  # normalizadas entre 0 y 1, es lo que pide WBF
        scores = r.boxes.conf.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy()

        if tta:
            rv = modelo.predict(cv2.flip(img, 1), conf=0.001, verbose=False)[0]
            cajas_v = rv.boxes.xyxyn.cpu().numpy()
            cajas_v[:, [0, 2]] = 1 - cajas_v[:, [2, 0]]  # deshacer el volteo: x1 = 1 - x2, x2 = 1 - x1
            cajas, scores, cls = weighted_boxes_fusion(
                [cajas, cajas_v], [scores, rv.boxes.conf.cpu().numpy()], [cls, rv.boxes.cls.cpu().numpy()],
                iou_thr=0.55, skip_box_thr=0.001)

        for b, s, c in zip(cajas, scores, cls):
            preds.append(dict(img=f.stem, cls=clases[int(c)], box=b * [W, H, W, H], score=float(s)))
    return gts, preds


for tta in [False, True]:
    t0 = time.time()
    gts, preds = predecir(baseline, "val", tta=tta)
    ms = (time.time() - t0) / len({g["img"] for g in gts}) * 1000
    print("TTA" if tta else "sin TTA", " mAP50:", round(mv.map50(gts, preds), 3),
          " mAP50-95:", round(mv.map_coco(gts, preds), 3), f" {ms:.0f} ms/img")

## Modelo final

Juntamos los cambios que mejoraron en val (mirar la tabla de arriba y quitar los que no). Si el TTA mejoro, tambien se usa al evaluar el final.

In [ ]:
final = YOLO("yolov10s.pt")
final.train(data=str(DATA_DIR / "data.yaml"), seed=SEED, name="final", epochs=200, box=10, dfl=2, imgsz=960)

final = YOLO(Path(final.trainer.save_dir) / "weights" / "best.pt")
evaluar("final", final)

## Evaluacion en test

Ahora si usamos test, solo para el baseline y el modelo final.

In [ ]:
for nombre, modelo in [("baseline", baseline), ("final", final)]:
    m = modelo.val(data=str(DATA_DIR / "data.yaml"), split="test", name=nombre + "_test")
    print(nombre, "mAP50:", round(m.box.map50, 3), "mAP50-95:", round(m.box.map, 3),
          "ms/img:", round(m.speed["inference"], 1))

# el final sin TTA y con TTA, los dos medidos con mapviz para que se puedan comparar
for tta in [False, True]:
    gts, preds = predecir(final, "test", tta=tta)
    print("final", "con TTA" if tta else "sin TTA", " mAP50:", round(mv.map50(gts, preds), 3),
          "mAP50-95:", round(mv.map_coco(gts, preds), 3))

## Analisis TIDE antes y despues

El mismo analisis que en el baseline (media de los 10 umbrales de IoU = mAP50-95), para ver si ha bajado el error de localizacion.

In [ ]:
umbrales = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
tipos = ["loc", "cls", "dup", "bg", "miss"]
nombres = ["localizacion", "clasificacion", "duplicados", "fondo", "no detectadas"]

def tide(modelo):
    gts, preds = predecir(modelo, "test")
    res = [mv.error_breakdown(gts, preds, iou_thr=t) for t in umbrales]
    return [np.mean([r["delta"][k] for r in res]) for k in tipos]

gan_base = tide(baseline)
gan_final = tide(final)
for n, a, b in zip(nombres, gan_base, gan_final):
    print(f"{n:15s} baseline {a:.3f}   final {b:.3f}")

y = np.arange(len(tipos))
plt.barh(y - 0.2, gan_base, 0.4, label="baseline")
plt.barh(y + 0.2, gan_final, 0.4, label="final")
plt.yticks(y, nombres)
plt.gca().invert_yaxis()
plt.xlabel("mAP50-95 ganado si se arregla ese error (test)")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusiones

(rellenar con los resultados)